In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Find the root from the notebook's current working directory
candidates = [Path.cwd(), *Path.cwd().parents]

PROJECT_ROOT = next(
    (
        path for path in candidates
        if (path / "dataset" / "X_scaled.csv").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing dataset/X_scaled.csv. "
    )

FEATURES_PATH = PROJECT_ROOT / "dataset" / "X_scaled.csv"

LABEL_PATHS = {
    "DEC Frozen": PROJECT_ROOT / "final_dec" / "model" / "final_labels.npy",
    "DEC Trainable": (
        PROJECT_ROOT / "experiments" / "encoder_ablation"
        / "runs" / "trainable" / "full_labels.npy"
    ),
    "IDEC": PROJECT_ROOT / "final_idec" / "model" / "final_labels.npy",
}

print("Project root:", PROJECT_ROOT)
print("Feature file exists:", FEATURES_PATH.is_file())

for model_name, path in LABEL_PATHS.items():
    print(f"{model_name}: {path.is_file()} — {path}")


In [2]:
X = pd.read_csv(FEATURES_PATH).to_numpy(dtype=np.float64)

print("Feature matrix shape:", X.shape)
print("Missing values:", np.isnan(X).sum())
print("Infinite values:", np.isinf(X).sum())

assert X.shape == (99_997, 7), (
    f"Unexpected feature shape: {X.shape}"
)
assert np.isfinite(X).all(), "Features contain NaN or infinite values."

print("Feature validation passed.")


Feature matrix shape: (99997, 7)
Missing values: 0
Infinite values: 0
Feature validation passed.


In [3]:
labels_by_model = {}

for model_name, path in LABEL_PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing label file for {model_name}: {path}"
        )

    labels = np.load(path, allow_pickle=False)

    print(
        f"{model_name}: shape={labels.shape}, "
        f"dtype={labels.dtype}"
    )

    assert labels.ndim == 1, (
        f"{model_name}: expected a 1D label array."
    )
    assert len(labels) == len(X), (
        f"{model_name}: {len(labels)} labels for {len(X)} observations."
    )
    assert np.issubdtype(labels.dtype, np.integer), (
        f"{model_name}: labels should be integer cluster IDs."
    )

    cluster_ids, cluster_counts = np.unique(
        labels, return_counts=True
    )

    print("  Cluster IDs:", cluster_ids.tolist())
    print("  Cluster sizes:", cluster_counts.tolist())

    assert len(cluster_ids) == 5, (
        f"{model_name}: expected 5 non-empty clusters, "
        f"found {len(cluster_ids)}."
    )

    labels_by_model[model_name] = labels

print("Label shape and cluster-count checks passed.")


DEC Frozen: shape=(99997,), dtype=int64
  Cluster IDs: [0, 1, 2, 3, 4]
  Cluster sizes: [489, 28258, 20515, 25483, 25252]
DEC Trainable: shape=(99997,), dtype=int64
  Cluster IDs: [0, 1, 2, 3, 4]
  Cluster sizes: [929, 24030, 25683, 27174, 22181]
IDEC: shape=(99997,), dtype=int64
  Cluster IDs: [0, 1, 2, 3, 4]
  Cluster sizes: [492, 22916, 25806, 29657, 21126]
Label shape and cluster-count checks passed.


### Feature-Space Consistency Validation

Before comparing the clustering models, we verified that all models' cluster assignments correspond to the same 99,997 observations in the same row order.

- **DEC Frozen and IDEC:** Both load `dataset/X_scaled.csv` directly and pass the feature matrix to their respective training functions.
- **DEC Trainable:** Loads `embeddings/X_scaled.npy`. We verified that this matrix has shape `(99,997, 7)` and matches `dataset/X_scaled.csv` exactly, with a maximum absolute difference of `0.0`. Its inference procedure preserves input row order.
- **K-Means:** Will be fitted directly on `dataset/X_scaled.csv`, the common seven-dimensional standardized feature matrix.

Therefore, the four models' cluster assignments can be evaluated against the same feature space, independently of any latent representations learned internally by DEC and IDEC.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import KMeans

# Find the root
root = Path.cwd().parents[1]

# Load the common standardized feature matrix
X = pd.read_csv(root / "dataset" / "X_scaled.csv").to_numpy()

# Fit K-Means
kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=20
)

kmeans_labels = kmeans.fit_predict(X)

# Save labels
output_dir = root / "experiments" / "baseline_comparison" / "results"
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / "kmeans_labels.npy", kmeans_labels)

# Verify the output
unique, counts = np.unique(kmeans_labels, return_counts=True)

print("Feature matrix shape:", X.shape)
print("Labels shape:", kmeans_labels.shape)
print("Number of clusters:", len(unique))
print("Cluster sizes:", dict(zip(unique.tolist(), counts.tolist())))
print("Labels saved to:", output_dir / "kmeans_labels.npy")

c:\uni-generall\term8\پروژه\Roommate-Compatibility-Analysis\dec_env\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\uni-generall\term8\پروژه\Roommate-Compatibility-Analysis\dec_env\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\uni-generall\term8\پروژه\Roommate-Compatibility-Analysis\dec_env\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\hp\AppData\Local\Programs\Py

Feature matrix shape: (99997, 7)
Labels shape: (99997,)
Number of clusters: 5
Cluster sizes: {0: 26217, 1: 535, 2: 21153, 3: 24103, 4: 27989}
Labels saved to: c:\uni-generall\term8\پروژه\Roommate-Compatibility-Analysis\experiments\baseline_comparison\results\kmeans_labels.npy


In [3]:
# Load the existing labels
labels_by_model = {
    "K-Means": kmeans_labels,
    "DEC Frozen": np.load(root / "final_dec" / "model" / "final_labels.npy"),
    "DEC Trainable": np.load(
        root / "experiments" / "encoder_ablation" / "runs"
        / "trainable" / "full_labels.npy"
    ),
    "IDEC": np.load(root / "final_idec" / "model" / "final_labels.npy"),
}

# Validate labels
assert X.shape == (99997, 7), f"Unexpected feature shape: {X.shape}"

for name, labels in labels_by_model.items():
    assert labels.shape == (len(X),), (
        f"{name}: expected {len(X)} labels, got {labels.shape}"
    )
    assert np.isfinite(labels).all(), f"{name}: labels contain non-finite values"
    assert len(np.unique(labels)) == 5, (
        f"{name}: expected 5 clusters, got {len(np.unique(labels))}"
    )

print("All four models passed the label validation.")
print("All models have 99,997 labels and five nonempty clusters.")

All four models passed the label validation.
All models have 99,997 labels and five nonempty clusters.


In [5]:
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)

results = []

for name, labels in labels_by_model.items():
    print(f"Evaluating {name}...")

    results.append({
        "Model": name,
        "Silhouette (full data)": silhouette_score(X, labels),
        "Davies-Bouldin (full data)": davies_bouldin_score(X, labels),
        "Calinski-Harabasz (full data)": calinski_harabasz_score(X, labels),
    })

metrics_df = pd.DataFrame(results)

display(metrics_df.round(4))

Evaluating K-Means on all 99,997 observations...
Evaluating DEC Frozen on all 99,997 observations...
Evaluating DEC Trainable on all 99,997 observations...
Evaluating IDEC on all 99,997 observations...


,Model,Silhouette (full data),Davies-Bouldin (full data),Calinski-Harabasz (full data)
0,K-Means,0.1226,1.6786,16509.5876
1,DEC Frozen,0.0896,2.0025,13604.6264
2,DEC Trainable,0.0754,2.1870,11645.2335
3,IDEC,0.0794,2.0763,13256.0439


In [6]:
# Save metric comparison
metrics_path = output_dir / "metrics_comparison.csv"
metrics_df.to_csv(metrics_path, index=False)

# Save cluster sizes
cluster_counts_df = pd.DataFrame({
    name: pd.Series(labels).value_counts().sort_index()
    for name, labels in labels_by_model.items()
}).fillna(0).astype(int)

cluster_counts_path = output_dir / "cluster_counts.csv"
cluster_counts_df.to_csv(cluster_counts_path, index_label="Cluster")

print("Metrics saved to:", metrics_path)
print("Cluster counts saved to:", cluster_counts_path)

display(metrics_df.round(4))
display(cluster_counts_df)

Metrics saved to: c:\uni-generall\term8\پروژه\Roommate-Compatibility-Analysis\experiments\baseline_comparison\results\metrics_comparison.csv
Cluster counts saved to: c:\uni-generall\term8\پروژه\Roommate-Compatibility-Analysis\experiments\baseline_comparison\results\cluster_counts.csv


,Model,Silhouette (full data),Davies-Bouldin (full data),Calinski-Harabasz (full data)
0,K-Means,0.1226,1.6786,16509.5876
1,DEC Frozen,0.0896,2.0025,13604.6264
2,DEC Trainable,0.0754,2.1870,11645.2335
3,IDEC,0.0794,2.0763,13256.0439


,K-Means,DEC Frozen,DEC Trainable,IDEC
0,26217,489,929,492
1,535,28258,24030,22916
2,21153,20515,25683,25806
3,24103,25483,27174,29657
4,27989,25252,22181,21126


### Interpretation

When evaluated using the same seven-dimensional standardized feature space and all 99,997 observations, K-Means achieved the most favorable values across all three internal clustering metrics. DEC Frozen, DEC Trainable, and IDEC obtained lower Silhouette and Calinski–Harabasz scores and higher Davies–Bouldin indices. These findings indicate that the deep clustering assignments did not outperform the K-Means baseline according to these metrics in the original feature space.

This comparison evaluates cluster assignments in the original feature space, whereas the previously reported DEC/IDEC results were calculated in the learned latent space. The results therefore highlight the importance of the evaluation space when interpreting clustering quality; the two sets of scores should be reported separately.